In [3]:
import subprocess, sys
# Auto-install missing dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "selenium", "webdriver-manager"])
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
# ── 1. Launch headless Chrome (auto-downloads the right ChromeDriver) ─────────
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver.get("https://result.election.gov.np/PRVoteChartResult2082.aspx")
# ── 2. Wait until KnockoutJS has rendered all party rows ──────────────────────
wait = WebDriverWait(driver, 20)
wait.until(lambda d: len(d.find_elements(
    By.CSS_SELECTOR, '[data-bind="text: PoliticalPartyName"]')) >= 10)
# ── 3. Scrape party names, votes, and symbol IDs from the rendered DOM ─────────
name_els   = driver.find_elements(By.CSS_SELECTOR, '[data-bind="text: PoliticalPartyName"]')
vote_els   = driver.find_elements(By.CSS_SELECTOR, '[data-bind="text: convertNepNumber(commafy(TotWin))"]')
symbol_els = driver.find_elements(By.CSS_SELECTOR, '[data-bind*="SymbolID"]')
def nepali_to_int(s):
    """Convert Nepali numeral string '५१,८३,४९३' → 5183493"""
    return int(s.replace(',', '').translate(str.maketrans('०१२३४५६७८९', '0123456789')))
records = []
for name_el, vote_el, sym_el in zip(name_els, vote_els, symbol_els):
    sym_url = sym_el.get_attribute("src") or ""
    records.append({
        "party_name"      : name_el.text.strip(),
        "total_votes"     : nepali_to_int(vote_el.text.strip()),
        "symbol_id"       : sym_url.split("/")[-1].split(".")[0] if sym_url else None,
        "symbol_image_url": sym_url,
    })
driver.quit()
# ── 4. Build DataFrame ────────────────────────────────────────────────────────
df = (pd.DataFrame(records)
        .sort_values("total_votes", ascending=False)
        .reset_index(drop=True))
df.index += 1  # rank from 1
print(f"Parties: {len(df)}  |  Total votes: {df['total_votes'].sum():,}")
df

Parties: 57  |  Total votes: 10,835,025


,party_name,total_votes,symbol_id,symbol_image_url
1,राष्ट्रिय स्वतन्त्र पार्टी,5183493,2528,https://result.election.gov.np/Images/symbol-h...
2,नेपाली काँग्रेस,1759172,2583,https://result.election.gov.np/Images/symbol-h...
3,नेपाल कम्युनिष्ट पार्टी (एकीकृत मार्क्सवादी ले...,1455885,2598,https://result.election.gov.np/Images/symbol-h...
4,नेपाली कम्युनिष्ट पार्टी,811577,2557,https://result.election.gov.np/Images/symbol-h...
5,श्रम संस्कृति पार्टी,385902,2501,https://result.election.gov.np/Images/symbol-h...
6,राष्ट्रिय प्रजातन्त्र पार्टी,330684,2604,https://result.election.gov.np/Images/symbol-h...
7,"जनता समाजवादी पार्टी, नेपाल",182285,2542,https://result.election.gov.np/Images/symbol-h...
8,राष्ट्रिय परिवर्तन पार्टी,172489,2568,https://result.election.gov.np/Images/symbol-h...
9,जनमत पार्टी,79435,2585,https://result.election.gov.np/Images/symbol-h...
10,एकल चिन्ह चकिया (जाँतो) (राष्ट्रिय मुक्ति पार्...,62069,2531,https://result.election.gov.np/Images/symbol-h...


In [4]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "Codes" else CURRENT_DIR
EXPORT_DIR = PROJECT_DIR / "Exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = EXPORT_DIR / "proportional_votes.csv"
json_path = EXPORT_DIR / "proportional_votes.json"

df.to_csv(csv_path, index_label="rank")
df.to_json(json_path, orient="records", force_ascii=False, indent=2)
print(f"Exported {csv_path.name} and {json_path.name} to {EXPORT_DIR}")

Exported proportional_votes.csv and proportional_votes.json to /Users/bikki/Desktop/Election82_data_analysis/Exports
